# 03 · Building the database

The cleaned records go into a SQLite **star schema** (`sql/schema.sql`):

```
                dim_date ──┐            ┌── dim_brand
   dim_source ──┐          │            │
                ├── fact_price ─────────┼── dim_product
   dim_location ┘          │            └── dim_category
                └── fact_availability

   dim_cpi_item ── fact_cpi_benchmark  (kept apart on purpose)
```

**Grain of `fact_price`:** one price = one source × product × place × observation date × price type (retail / wholesale).

The load is **incremental**: a raw file whose fingerprint is already in `etl_raw_file` is skipped, so a new collection date is simply appended.

In [1]:
import sys, sqlite3
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 45)

import load
from config import DB_PATH

run_id, n_files, n_new, n_upd, missing = load.run()
print(f"run {run_id}: {n_files} new raw files, {n_new} new rows, {n_upd} updated rows")

run 5: 0 new raw files, 0 new rows, 0 updated rows


Running it a second time proves the incremental behaviour — nothing is reloaded:

In [2]:
print(load.run()[1:4])

(0, 0, 0)


## What ended up in each table

In [3]:
con = sqlite3.connect(DB_PATH)
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", con)["name"]
pd.DataFrame({"table": tables,
              "rows": [con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tables]})

,table,rows
0,bridge_commodity_cpi,39
1,dim_brand,234
2,dim_brand_alias,257
3,dim_category,26
4,dim_cpi_item,383
5,dim_date,1461
6,dim_location,204
7,dim_product,419
8,dim_product_name_history,397
9,dim_source,3


## Provenance: which raw file produced what

In [4]:
pd.read_sql("""SELECT source_code, relative_path, substr(sha256,1,16) AS sha256_prefix,
                      collection_date, records_parsed, loaded_at_utc
               FROM etl_raw_file ORDER BY raw_file_id""", con)

,source_code,relative_path,sha256_prefix,collection_date,records_parsed,loaded_at_utc
0,DOCA_PMD,data/raw/doca_dashboard/2026-10-01/doca_d...,88bd44fefe06e302,2026-10-01,275,2026-10-01T11:51:14Z
1,MOSPI_CPI,data/raw/mospi_cpi/2026-10-01/mospi_cpi_r...,0d4435dcb18cc7b1,2026-10-01,9107,2026-10-01T11:51:15Z
2,MOSPI_CPI,data/raw/mospi_cpi/2026-10-01/mospi_cpi_r...,ace37019545800d2,2026-10-01,728,2026-10-01T11:51:15Z
3,OPEN_PRICES,data/raw/open_prices/2026-10-01/openprice...,1bb2ed0abcf53c56,2026-10-01,402,2026-10-01T11:51:15Z


## A typical star-schema query

Latest official retail prices for pulses, normalised per kg:

In [5]:
pd.read_sql("""
SELECT p.product_name, d.full_date, f.price_paid, p.pack_size_label,
       ROUND(f.normalized_price, 2) AS price_per_kg
FROM fact_price f
JOIN dim_product  p ON p.product_key  = f.product_key
JOIN dim_category c ON c.category_key = f.category_key
JOIN dim_date     d ON d.date_key     = f.date_key
WHERE c.category = 'Pulses' AND f.price_type = 'retail'
  AND f.lookback_label = 'Price As On' AND f.include_in_analysis = 1
ORDER BY price_per_kg DESC""", con)

,product_name,full_date,price_paid,pack_size_label,price_per_kg
0,Tur/Arhar Dal,2026-09-30,125.81,1 Kg,125.81
1,Urad Dal,2026-09-30,123.18,1 Kg,123.18
2,Moong Dal,2026-09-30,112.11,1 Kg,112.11
3,Besan,2026-09-30,96.63,1 Kg,96.63
4,Masoor Dal,2026-09-30,90.38,1 Kg,90.38
5,Gram Dal,2026-09-30,88.96,1 Kg,88.96
6,Gram Dal (Bharat Dal adjusted),2026-09-30,84.61,1 Kg,84.61


## Design choices worth explaining

- **Two date keys** on the fact (`date_key` observation, `collection_date_key` capture).
- **Natural key** `(source, source_record_id)` — for DoCA it's `COMMODITY|date|retail`. Re-capturing the same date updates it; if the value changed (weekend prices are provisional) the old value goes to `fact_price_revision`.
- **CPI is a different measure** (an index, not a price), so it has its own fact table and only meets the price data through `bridge_commodity_cpi` for side-by-side comparison.

Next: **04 · Data quality**.

In [6]:
con.close()